In [31]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.preprocessing import OneHotEncoder
from ipywidgets import interact
from ipywidgets import widgets
import os
os.chdir("E:/Machine_Learning/Datasets/")

In [32]:
housing = pd.read_csv("Housing.csv")
housing

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...
541,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


In [34]:
intervals = [(0,50000),(50000,100000),(100000,150000),(150000,200000)]
bins = pd.IntervalIndex.from_tuples(intervals)
housing['price_slab']=pd.cut(housing['price'],bins)
housing['price_slab']

0            (0, 50000]
1            (0, 50000]
2            (0, 50000]
3       (50000, 100000]
4       (50000, 100000]
             ...       
541     (50000, 100000]
542     (50000, 100000]
543    (100000, 150000]
544    (100000, 150000]
545    (100000, 150000]
Name: price_slab, Length: 546, dtype: category
Categories (4, interval[int64, right]): [(0, 50000] < (50000, 100000] < (100000, 150000] < (150000, 200000]]

In [35]:
intervals = [(0,5000),(5000,10000),(10000,15000),(15000,20000)]
bins = pd.IntervalIndex.from_tuples(intervals)
housing['area_slab']=pd.cut(housing['lotsize'],bins)
housing['area_slab']

0      (5000, 10000]
1          (0, 5000]
2          (0, 5000]
3      (5000, 10000]
4      (5000, 10000]
           ...      
541        (0, 5000]
542    (5000, 10000]
543    (5000, 10000]
544    (5000, 10000]
545    (5000, 10000]
Name: area_slab, Length: 546, dtype: category
Categories (4, interval[int64, right]): [(0, 5000] < (5000, 10000] < (10000, 15000] < (15000, 20000]]

In [36]:
housing.drop(['price','lotsize'],axis = 1,inplace = True)

In [37]:
ohc = OneHotEncoder(sparse_output=False,dtype=np.bool_).set_output(transform='pandas')
fp_df = ohc.fit_transform(housing)
fp_df.head()

,bedrooms_1,bedrooms_2,bedrooms_3,bedrooms_4,bedrooms_5,bedrooms_6,bathrms_1,bathrms_2,bathrms_3,bathrms_4,...,prefarea_no,prefarea_yes,"price_slab_(0, 50000]","price_slab_(50000, 100000]","price_slab_(100000, 150000]","price_slab_(150000, 200000]","area_slab_(0, 5000]","area_slab_(5000, 10000]","area_slab_(10000, 15000]","area_slab_(15000, 20000]"
0,False,False,True,False,False,False,True,False,False,False,...,True,False,True,False,False,False,False,True,False,False
1,False,True,False,False,False,False,True,False,False,False,...,True,False,True,False,False,False,True,False,False,False
2,False,False,True,False,False,False,True,False,False,False,...,True,False,True,False,False,False,True,False,False,False
3,False,False,True,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
4,False,True,False,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False


In [38]:
def gen_rules(min_support , min_confidence):
    itemsets = apriori(fp_df, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets , metric='confidence' , min_threshold=min_confidence)
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift',]]
    relv_rules= rules[rules['lift']>1]
    display(relv_rules.sort_values(by=['confidence', 'lift', 'support'], ascending=False))

In [39]:
gen_rules(min_support=0.6,min_confidence=0.7)

,antecedents,consequents,support,confidence,lift
23,"(driveway_yes, bathrms_1)",(gashw_no),0.602564,0.964809,1.011105
5,(bathrms_1),(gashw_no),0.708791,0.962687,1.008881
29,"(bathrms_1, recroom_no)",(gashw_no),0.602564,0.959184,1.005210
8,(driveway_yes),(gashw_no),0.820513,0.955224,1.001060
33,"(driveway_yes, recroom_no)",(gashw_no),0.663004,0.955145,1.000977
18,(fullbase_no),(gashw_no),0.620879,0.954930,1.000752
13,(fullbase_no),(recroom_no),0.602564,0.926761,1.126974
9,(gashw_no),(driveway_yes),0.820513,0.859885,1.001060
37,"(prefarea_no, gashw_no)",(recroom_no),0.622711,0.858586,1.044071
16,(prefarea_no),(recroom_no),0.655678,0.856459,1.041485


In [40]:
interact(
    gen_rules , 
    min_support = widgets.FloatSlider(value = 0.05 , min = 0.01, max = 1 , description = 'Min Support:'),
    min_confidence = widgets.FloatSlider(value = 0.05 , min = 0.01, max = 1 , description = 'Min Confidence:')
)

interactive(children=(FloatSlider(value=0.05, description='Min Support:', max=1.0, min=0.01), FloatSlider(valu…

<function __main__.gen_rules(min_support, min_confidence)>

----
# Same process on human resource analytics
----

In [42]:
os.chdir('E:/Machine_Learning/Cases/human-resources-analytics')

In [43]:
df = pd.read_csv('HR_comma_sep.csv')
df

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [44]:
df.describe()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years
count,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000
mean,0.612890,0.716108,3.803001,201.050150,3.498166,0.144648,0.237879,0.021274
std,0.248615,0.171166,1.232266,49.943356,1.460262,0.351758,0.425799,0.144300
min,0.090000,0.360000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000
25%,0.440000,0.560000,3.000000,156.000000,3.000000,0.000000,0.000000,0.000000
50%,0.640000,0.720000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000
75%,0.820000,0.870000,5.000000,245.000000,4.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000


In [45]:
def binning(input,b):
    binned = pd.cut(input,bins=b,)
    return binned

In [46]:
df['satisfaction'] = binning(df['satisfaction_level'],5)
df['evaluation'] = binning(df['last_evaluation'],5)
df['projects'] = binning(df['number_project'],3)
df['avg_hours'] = binning(df['average_montly_hours'],5)
df['years_in_comp'] = binning(df['time_spend_company'],3)

In [48]:
df.drop(['satisfaction_level','last_evaluation','number_project','average_montly_hours','time_spend_company'],axis=1,inplace=True)

In [51]:
ohe = OneHotEncoder(sparse_output=False,dtype=np.bool_).set_output(transform='pandas')
fp_df = ohe.fit_transform(df)
fp_df

,Work_accident_0,Work_accident_1,left_0,left_1,promotion_last_5years_0,promotion_last_5years_1,Department_IT,Department_RandD,Department_accounting,Department_hr,...,"projects_(3.667, 5.333]","projects_(5.333, 7.0]","avg_hours_(95.786, 138.8]","avg_hours_(138.8, 181.6]","avg_hours_(181.6, 224.4]","avg_hours_(224.4, 267.2]","avg_hours_(267.2, 310.0]","years_in_comp_(1.992, 4.667]","years_in_comp_(4.667, 7.333]","years_in_comp_(7.333, 10.0]"
0,True,False,False,True,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
1,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,False,True,False,False,True,False
2,True,False,False,True,True,False,False,False,False,False,...,False,True,False,False,False,True,False,True,False,False
3,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,False,True,False,False,True,False
4,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14990,True,False,False,True,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
14991,True,False,False,True,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
14992,True,False,False,True,True,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
14993,True,False,False,True,True,False,False,False,False,False,...,False,True,False,False,False,False,True,True,False,False


In [57]:
def gen_rules(min_support , min_confidence):
    itemsets = apriori(fp_df, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets , metric='confidence' , min_threshold=min_confidence)
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift',]]
    relv_rules= rules[rules['lift']>1]
    display(relv_rules.sort_values(by=['confidence', 'lift', 'support'], ascending=False))

In [58]:
gen_rules(0.2,0.5)

,antecedents,consequents,support,confidence,lift
86,"(Work_accident_0, left_1)",(promotion_last_5years_0),0.225609,0.995586,1.017226
34,(left_1),(promotion_last_5years_0),0.236612,0.994673,1.016294
374,"(left_0, salary_low, Work_accident_0, years_in...",(promotion_last_5years_0),0.244281,0.993491,1.015085
301,"(salary_low, Work_accident_0, years_in_comp_(1...",(promotion_last_5years_0),0.341581,0.993213,1.014802
203,"(projects_(3.667, 5.333], salary_low)",(promotion_last_5years_0),0.225275,0.992654,1.014230
...,...,...,...,...,...
424,"(left_0, Work_accident_0, years_in_comp_(1.992...","(promotion_last_5years_0, projects_(3.667, 5.3...",0.284428,0.528894,1.139805
415,"(projects_(1.995, 3.667])","(promotion_last_5years_0, left_0, Work_acciden...",0.226809,0.528024,1.001355
368,"(left_0, years_in_comp_(1.992, 4.667])","(promotion_last_5years_0, projects_(3.667, 5.3...",0.340914,0.526197,1.133992
253,"(left_0, Work_accident_0)","(promotion_last_5years_0, projects_(3.667, 5.3...",0.328843,0.523017,1.127139
